# Multipartitos de período 3 contra el globo

Compara $t_2$ de los multipartitos óptimos con un puente (esqueleto $K(a_1,\dots,a_p)$ de $n-1$ vértices, $s$ en la parte $a_1$ y $t$ una hoja pegada a otro vértice $v$ de esa misma parte) con $t_2$ del grafo globo, para $m=\binom{n-2}{2}+1$ y $m=\binom{n-2}{2}+2$.

Las partes siguen un patrón periódico de período $3$ en $n$ (con $k=n-1$ vértices en el esqueleto):

| $n \bmod 3$ | $m=\binom{n-2}{2}+1$ | $m=\binom{n-2}{2}+2$ |
|---|---|---|
| $0$ | $K(3_{st},3,\dots,3,2)$ | $K(3_{st},3,\dots,3,1,1)$ |
| $1$ | $K(4_{st},3,\dots,3,2,2,1)$ | $K(3_{st},3,\dots,3,2,1)$ |
| $2$ | $K(3_{st},3,\dots,3,1)$ | $K(3_{st},3,\dots,3,2,2)$ |

In [1]:
import os
while not os.path.isdir("DataTypes"):
    os.chdir("..")

load("DataTypes/Grafo2Terminales.sage")
load("Scripts/NumeroDe2Bosques.sage")
load("Scripts/GrafoBalloon.sage")

In [2]:
INICIO, FIN = 800, 900      # rango de n (ambos inclusive)
FIN_VERIFICACION = 50      # hasta este n se contrasta la fórmula cerrada con el determinante

def ValoresDeM(n):
    return [binomial(n - 2, 2) + 1, binomial(n - 2, 2) + 2]

## Partes del patrón y fórmula cerrada

In [3]:
def PartesPatron(n, indice):
    '''
        Partes del multipartito (la primera contiene a s y a v) para m = C(n-2,2) + 1 + indice,
        o None si el patrón no existe para ese n (cantidad negativa de partes de tamaño 3).
    '''
    k = n - 1
    r = n % 3
    if r == 0:
        cola = [[2], [1, 1]][indice]
        partes = [3] * ((k - sum(cola)) // 3) + cola
    elif r == 1:
        if indice == 0:
            if k < 9:
                return None
            partes = [4] + [3] * ((k - 9) // 3) + [2, 2, 1]
        else:
            partes = [3] * ((k - 3) // 3) + [2, 1]
    else:
        cola = [[1], [2, 2]][indice]
        partes = [3] * ((k - sum(cola)) // 3) + cola
    if min(partes) < 1 or sum(partes) != k or partes[0] < 3:
        return None
    return partes

def NombrePartes(partes):
    resto = []
    for p in partes[1:]:
        if resto and resto[-1][0] == p:
            resto[-1][1] += 1
        else:
            resto.append([p, 1])
    return "K(" + ",".join([f"{partes[0]}_st"] + [str(p) if r == 1 else f"{p}^{r}" for p, r in resto]) + ") + 1 puente"

def T2Multipartito(partes):
    '''
        t2 del grafo formado por el multipartito completo de `partes` (esqueleto, k vértices) más una
        hoja t pegada a un vértice v de la primera parte, con s otro vértice de esa parte.
    '''
    k = sum(partes)
    p = len(partes)
    arboles = QQ(k) ** (p - 2) * prod(QQ(k - a) ** (a - 1) for a in partes)
    b = [partes[0] - 2] + list(partes[1:])
    d = [k - a for a in partes]
    c = [di + bi for di, bi in zip(d, b)]
    dos_bosques = prod(QQ(di) ** (bi - 1) for di, bi in zip(d, b)) * prod(c) * (1 - sum(QQ(bi) / ci for bi, ci in zip(b, c)))
    valor = arboles + dos_bosques
    assert valor in ZZ
    return ZZ(valor)

def GrafoMultipartito(partes):
    '''
        Retorna Grafo2Terminales: multipartito de `partes` más una hoja t pegada a v, con s y v
        en la primera parte.
    '''
    H = graphs.CompleteMultipartiteGraph(partes)
    k = H.order()
    s, v = 0, 1                      # los vértices 0..partes[0]-1 forman la primera parte
    H.add_edge(v, k)
    return Grafo2Terminales(H, s, k)

In [4]:
# Verificación de la fórmula cerrada y del patrón contra el determinante
for n in range(INICIO, FIN_VERIFICACION + 1):
    for indice, m in enumerate(ValoresDeM(n)):
        partes = PartesPatron(n, indice)
        if partes is None:
            continue
        TC = GrafoMultipartito(partes)
        assert TC.grafo.order() == n and TC.grafo.size() == m, (n, m, partes)
        assert NumeroDe2Bosques(TC) == T2Multipartito(partes), (n, m, partes)
print("Fórmula cerrada y patrón verificados para n <=", FIN_VERIFICACION)

Fórmula cerrada y patrón verificados para n <= 50


## Comparación con el globo

In [5]:
resultados = []
for n in range(INICIO, FIN + 1):
    for indice, m in enumerate(ValoresDeM(n)):
        partes = PartesPatron(n, indice)
        t2_globo = ZZ(NumeroDe2Bosques(ConstruirGrafo2TerminalesBalloon(n, m)))
        if partes is None:
            print(f"({n}, {m}): sin multipartito del patrón")
            continue
        t2_multi = T2Multipartito(partes)
        cumple = t2_multi > t2_globo
        resultados.append((n, m, cumple))
        print(f"({n}, {m}): {NombrePartes(partes)}: t2 = {t2_multi}, t2(globo) = {t2_globo}, "
              f"t2/t2(globo) = {float(t2_multi / t2_globo):.6f} -> {'SE CUMPLE' if cumple else 'NO SE CUMPLE'}")

(800, 318004): K(3_st,3^265,1) + 1 puente: t2 = 2895598735229719901655669737713375630615403307448346970842776262252181056016228703991252072712599654862828109419511770366192987630489440777685498179652976400241018953456057203223494180415788369122832077696743606839625309300910161944737848059865552482361570383846463196898531203393111638777620821284553980222537460685396625505645830911548361069462391781123846216115281175142063480708704834920121108790223189095210427857537555008288073882387038422974406413541150673906359145989042479381771814981080116561669729928819390966753052879246894493537192830179233554945780259266263646679461138540284811249873596423518744321060620667596120759628223508618608127135968459423706816402350169256184289342264870496804235310614311991505617084881390470142871409246347199784294405340282825547697716816294535414569118718546537860988877139449881133470493997266379806857281326106008314423959027476341063595580895940705861394561927022020992004859447489246320971711099253599109

KeyboardInterrupt: 

In [14]:
fallan = [(n, m) for n, m, cumple in resultados if not cumple]
print(f"Pares (n,m) evaluados: {len(resultados)}")
print("La conjetura se cumple en todos." if not fallan else f"No se cumple en: {fallan}")

Pares (n,m) evaluados: 809
La conjetura se cumple en todos.
